In [1]:
import pandas as pd
import re

INPUT_FILE = "facility_names_from_df.csv"
OUTPUT_FILE = "facility_names_standardized.csv"
REVIEW_FILE = "facility_names_review_needed.csv"

# 讀取 CSV；utf-8-sig 可處理檔案開首的 BOM 字元
df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")

# 預設原始欄位叫 facility_name
df = df.rename(columns={df.columns[0]: "original_name"})
df["original_name"] = (
    df["original_name"]
    .fillna("")
    .astype(str)
    .str.replace("\u3000", " ", regex=False)
    .str.strip()
)

# 移除空白列
df = df[df["original_name"] != ""].copy()


def normalize_text(text):
    """統一常見字形、錯別字及多餘空白，但保留原始名稱於 original_name。"""
    x = text.strip()

    replacements = {
        "啞呤": "啞鈴",
        "杠鈴": "槓鈴",
        "鍛練": "鍛鍊",
        "鍛練器": "鍛鍊器",
        "鍛練機": "鍛鍊機",
        "訓練 器": "訓練器",
        "練習 器": "練習器",
        "靠背 式": "靠背式",
        "／": "/",
        "–": "-",
        "—": "-",
        "∘": "度",
    }

    for old, new in replacements.items():
        x = x.replace(old, new)

    # 將多個空白壓縮為一個
    x = re.sub(r"\s+", " ", x)

    # 移除括號前後多餘空白
    x = re.sub(r"\s*\(\s*", "(", x)
    x = re.sub(r"\s*\)\s*", ")", x)

    return x


def make_record(original_name):
    """
    回傳：
    standard_name, main_category, sub_category,
    resistance_type, primary_muscles, notes
    """
    name = normalize_text(original_name)
    note = ""

    # ---------------------------------------------------------
    # 1. 量度及評估器材
    # ---------------------------------------------------------
    if any(k in name for k in ["體重磅", "電子磅", "電子體磅", "身高尺"]):
        return {
            "standard_name": "體重磅連身高尺",
            "main_category": "體適能評估及量度器材",
            "sub_category": "體重及身高量度",
            "resistance_type": "不適用",
            "primary_muscles": "不適用",
            "notes": ""
        }

    # 「磅」本身過於模糊
    if name in ["磅", "重量磅", "電子重量磅"]:
        return {
            "standard_name": name,
            "main_category": "待覆核",
            "sub_category": "重量或體重量度",
            "resistance_type": "待確認",
            "primary_muscles": "不適用",
            "notes": "名稱過於簡略；需按器材相片或型號確認是槓片、體重磅或其他設備。"
        }

    # ---------------------------------------------------------
    # 2. 心肺訓練器材
    # ---------------------------------------------------------
    if "跑步機" in name or "步行機" in name:
        return {
            "standard_name": "跑步機",
            "main_category": "心肺訓練器材",
            "sub_category": "跑步類",
            "resistance_type": "心肺",
            "primary_muscles": "下肢／心肺",
            "notes": "電子／電動等字樣已整合。"
        }

    if any(k in name for k in ["划船機", "划艇機", "電子划船機", "手搖划艇機"]):
        std = "手搖划艇機" if "手搖" in name else "划船機"
        return {
            "standard_name": std,
            "main_category": "心肺訓練器材",
            "sub_category": "划艇類",
            "resistance_type": "心肺",
            "primary_muscles": "背部／手臂／下肢／心肺",
            "notes": ""
        }

    if any(k in name for k in ["交叉訓練", "橢圓", "心肺交叉"]):
        return {
            "standard_name": "橢圓交叉訓練機",
            "main_category": "心肺訓練器材",
            "sub_category": "橢圓／交叉訓練類",
            "resistance_type": "心肺",
            "primary_muscles": "下肢／心肺",
            "notes": ""
        }

    if any(k in name for k in ["台階機", "踏步機", "樓梯機"]):
        return {
            "standard_name": "踏步機／樓梯機",
            "main_category": "心肺訓練器材",
            "sub_category": "踏步／登樓類",
            "resistance_type": "心肺",
            "primary_muscles": "臀部／大腿／小腿／心肺",
            "notes": ""
        }

    if any(k in name for k in ["滑雪機", "滑雪運動機"]):
        return {
            "standard_name": "滑雪機",
            "main_category": "心肺訓練器材",
            "sub_category": "全身帶氧類",
            "resistance_type": "心肺",
            "primary_muscles": "全身／心肺",
            "notes": ""
        }

    if any(k in name for k in ["太空漫步", "滑行機", "弧步運轉", "全身運轉", "帶氧運轉", "健體運轉", "運轉機"]):
        return {
            "standard_name": "全身帶氧訓練機",
            "main_category": "心肺訓練器材",
            "sub_category": "全身帶氧類",
            "resistance_type": "心肺",
            "primary_muscles": "全身／心肺",
            "notes": "建議按相片或型號再細分。"
        }

    if any(k in name for k in ["單車", "腳踏車"]):
        if any(k in name for k in ["靠背", "背靠", "坐臥", "臥式"]):
            std = "臥式健身單車"
        elif any(k in name for k in ["坐式", "坐立式"]):
            std = "坐式健身單車"
        else:
            std = "直立式健身單車"

        return {
            "standard_name": std,
            "main_category": "心肺訓練器材",
            "sub_category": "單車類",
            "resistance_type": "心肺",
            "primary_muscles": "下肢／心肺",
            "notes": ""
        }

    # ---------------------------------------------------------
    # 3. 自由重量及配件
    # ---------------------------------------------------------
    if "啞鈴" in name:
        notes = ""
        weights = re.findall(r"[\d.]+\s*(?:公斤|kg|磅|lbs|LB)", name, flags=re.I)
        if weights:
            notes = "原始重量資料：" + "、".join(weights)

        return {
            "standard_name": "啞鈴",
            "main_category": "自由重量器材",
            "sub_category": "啞鈴",
            "resistance_type": "自由重量",
            "primary_muscles": "全身",
            "notes": notes
        }

    if "壺鈴" in name:
        return {
            "standard_name": "壺鈴",
            "main_category": "自由重量器材",
            "sub_category": "壺鈴",
            "resistance_type": "自由重量",
            "primary_muscles": "全身／核心",
            "notes": ""
        }

    if "藥球" in name:
        return {
            "standard_name": "藥球",
            "main_category": "自由重量器材",
            "sub_category": "藥球",
            "resistance_type": "自由重量",
            "primary_muscles": "全身／核心",
            "notes": ""
        }

    if "槓鈴" in name or "曲尺槓鈴" in name or "鐵餅" in name:
        std = "槓鈴及槓片"
        if "曲尺" in name:
            std = "曲尺槓鈴（EZ Bar）"

        return {
            "standard_name": std,
            "main_category": "自由重量器材",
            "sub_category": "槓鈴／槓片",
            "resistance_type": "自由重量",
            "primary_muscles": "全身",
            "notes": ""
        }

    # ---------------------------------------------------------
    # 4. 訓練椅、訓練板、羅馬椅
    # ---------------------------------------------------------
    if "羅馬椅" in name or "45度背肌伸展" in name:
        return {
            "standard_name": "羅馬椅（45度背肌伸展器）",
            "main_category": "訓練椅、訓練台及核心訓練板",
            "sub_category": "下背伸展",
            "resistance_type": "自體重",
            "primary_muscles": "豎脊肌／臀大肌／腿後肌",
            "notes": ""
        }

    if any(k in name for k in ["捲腹板", "仰臥起坐", "腹肌板", "收腹板", "摺腹板", "修腹板"]):
        return {
            "standard_name": "腹肌訓練板",
            "main_category": "訓練椅、訓練台及核心訓練板",
            "sub_category": "腹肌訓練板",
            "resistance_type": "自體重",
            "primary_muscles": "腹直肌／腹斜肌",
            "notes": ""
        }

    if any(k in name for k in ["訓練椅", "練習椅", "啞鈴椅", "健身椅", "平式椅", "長椅", "舉重練習椅"]):
        if any(k in name for k in ["可調", "調校", "調較", "多斜度", "上傾", "下傾"]):
            std = "可調式訓練椅"
        elif "平板" in name or "平式" in name:
            std = "平板訓練椅"
        else:
            std = "訓練椅"

        return {
            "standard_name": std,
            "main_category": "訓練椅、訓練台及核心訓練板",
            "sub_category": "訓練椅",
            "resistance_type": "配合自由重量／自體重",
            "primary_muscles": "依訓練動作而定",
            "notes": ""
        }

    # ---------------------------------------------------------
    # 5. 伸展、恢復
    # ---------------------------------------------------------
    if any(k in name for k in ["伸展", "拉伸", "拉筋", "拉根", "肌肉伸展"]):
        return {
            "standard_name": "伸展訓練器",
            "main_category": "柔軟度、伸展及復康類器材",
            "sub_category": "伸展器材",
            "resistance_type": "自體重／輕阻力",
            "primary_muscles": "全身",
            "notes": ""
        }

    if "按摩機" in name:
        return {
            "standard_name": "按摩機",
            "main_category": "柔軟度、伸展及復康類器材",
            "sub_category": "被動恢復",
            "resistance_type": "不適用",
            "primary_muscles": "不適用",
            "notes": ""
        }

    if "震動帶" in name:
        return {
            "standard_name": "震動帶",
            "main_category": "柔軟度、伸展及復康類器材",
            "sub_category": "被動恢復",
            "resistance_type": "震動",
            "primary_muscles": "依使用部位而定",
            "notes": ""
        }

    # ---------------------------------------------------------
    # 6. 輔助引體上升／雙槓
    # ---------------------------------------------------------
    if any(k in name for k in ["輔助引體", "力量輔助引體", "重量輔助", "引體向上/雙槓"]):
        return {
            "standard_name": "輔助引體上升／雙槓訓練機",
            "main_category": "輔助自重訓練器材",
            "sub_category": "引體上升／雙槓撐體",
            "resistance_type": "配重輔助",
            "primary_muscles": "背闊肌／二頭肌／三頭肌／胸肌／核心",
            "notes": ""
        }

    # 純自重架
    if any(k in name for k in ["槓上雙臂屈伸架", "提膝練習架", "肘寸掌提膝架", "單雙槓"]):
        return {
            "standard_name": "自重引體上升／雙槓訓練架",
            "main_category": "自由重量支架及訓練架",
            "sub_category": "引體上升／雙槓／提膝",
            "resistance_type": "自體重",
            "primary_muscles": "背部／胸部／手臂／核心",
            "notes": ""
        }

    # ---------------------------------------------------------
    # 7. 多功能滑輪及功能性訓練
    # ---------------------------------------------------------
    if any(k in name for k in [
        "雙滑輪", "多功能拉力", "十字交叉拉力", "高低滑輪",
        "多用途拉力", "多功能綜合", "雙軌式多功能",
        "功能性肌力", "上身肌力"
    ]):
        return {
            "standard_name": "多功能滑輪訓練器",
            "main_category": "滑輪、多功能及功能性訓練器",
            "sub_category": "可調式滑輪／綜合訓練",
            "resistance_type": "插銷配重／滑輪",
            "primary_muscles": "全身",
            "notes": "多功能器材；可配合不同附件訓練胸、背、肩、手臂、腿及核心。"
        }

    # ---------------------------------------------------------
    # 8. 核心及軀幹器材
    # ---------------------------------------------------------
    if any(k in name for k in ["扭腰", "轉腰", "腰部旋轉", "轉體", "旋體", "軀幹旋轉", "腹斜肌"]):
        return {
            "standard_name": "軀幹旋轉訓練機",
            "main_category": "核心、腹部、下背及軀幹旋轉器材",
            "sub_category": "軀幹旋轉／腹斜肌",
            "resistance_type": "插銷配重／槓桿式",
            "primary_muscles": "腹斜肌／腹橫肌／核心",
            "notes": ""
        }

    if any(k in name for k in ["提膝", "提腿", "抬腿", "懸垂"]):
        return {
            "standard_name": "提膝／提腿訓練器",
            "main_category": "核心、腹部、下背及軀幹旋轉器材",
            "sub_category": "提膝／下腹",
            "resistance_type": "自體重／配重",
            "primary_muscles": "腹直肌／髖屈肌／核心",
            "notes": ""
        }

    if any(k in name for k in ["收腹", "腹肌", "腹部屈曲", "捲腹"]):
        return {
            "standard_name": "腹肌訓練器",
            "main_category": "核心、腹部、下背及軀幹旋轉器材",
            "sub_category": "腹部屈曲／捲腹",
            "resistance_type": "插銷配重／槓桿式",
            "primary_muscles": "腹直肌／腹斜肌",
            "notes": ""
        }

    if any(k in name for k in [
        "背伸", "背部伸展", "背肌伸展", "下背伸展",
        "伸背", "背部屈伸", "背部伸張"
    ]):
        return {
            "standard_name": "背部伸展訓練器",
            "main_category": "核心、腹部、下背及軀幹旋轉器材",
            "sub_category": "下背伸展",
            "resistance_type": "插銷配重／槓桿式／自體重",
            "primary_muscles": "豎脊肌／臀大肌／腿後肌",
            "notes": ""
        }

    if any(k in name for k in ["腹/背", "腹部/背部", "收腹/伸背", "腹部前屈/背部伸展", "伸背/腹部"]):
        return {
            "standard_name": "腹背複合訓練器",
            "main_category": "核心、腹部、下背及軀幹旋轉器材",
            "sub_category": "腹部及下背複合",
            "resistance_type": "插銷配重／槓桿式",
            "primary_muscles": "腹直肌／腹斜肌／豎脊肌",
            "notes": "雙功能器材。"
        }

    # ---------------------------------------------------------
    # 9. 下肢器材
    # ---------------------------------------------------------
    if any(k in name for k in ["內收", "外展", "內外側", "外側/內側", "外/內側", "內彎", "外彎"]):
        return {
            "standard_name": "大腿內收／外展訓練器",
            "main_category": "固定式阻力訓練器械",
            "sub_category": "下肢－髖內收／外展",
            "resistance_type": "插銷配重／槓桿式",
            "primary_muscles": "大腿內收肌群／臀中肌／臀小肌",
            "notes": ""
        }

    if any(k in name for k in ["臀部", "臀肌", "髖部", "臀部伸展", "臀部收縮"]):
        return {
            "standard_name": "臀部／髖部訓練器",
            "main_category": "固定式阻力訓練器械",
            "sub_category": "下肢－臀部及髖關節",
            "resistance_type": "插銷配重／槓桿式",
            "primary_muscles": "臀大肌／臀中肌／髖屈肌",
            "notes": ""
        }

    if any(k in name for k in ["小腿伸展", "提踵"]):
        return {
            "standard_name": "小腿訓練器",
            "main_category": "固定式阻力訓練器械",
            "sub_category": "下肢－小腿",
            "resistance_type": "插銷配重／槓桿式",
            "primary_muscles": "腓腸肌／比目魚肌",
            "notes": "需按器材相片確認是小腿伸展或提踵型。"
        }

    if any(k in name for k in ["蹬腿", "推蹬", "撐腿", "推腿", "蹲腿"]):
        return {
            "standard_name": "腿推／蹬腿訓練機",
            "main_category": "固定式阻力訓練器械",
            "sub_category": "下肢－腿推／蹬腿",
            "resistance_type": "插銷配重／槓片式",
            "primary_muscles": "股四頭肌／臀大肌／腿後肌",
            "notes": ""
        }

    if any(k in name for k in ["腿彎舉", "屈腿", "後腿屈", "後大腿屈", "大腿後肌", "腿部彎曲"]):
        return {
            "standard_name": "腿彎舉訓練機",
            "main_category": "固定式阻力訓練器械",
            "sub_category": "下肢－腿後肌",
            "resistance_type": "插銷配重／槓桿式",
            "primary_muscles": "腿後肌群／腓腸肌",
            "notes": ""
        }

    if any(k in name for k in ["腿伸展", "大腿伸展", "前大腿", "四頭肌"]):
        return {
            "standard_name": "腿伸展訓練機",
            "main_category": "固定式阻力訓練器械",
            "sub_category": "下肢－股四頭肌",
            "resistance_type": "插銷配重／槓桿式",
            "primary_muscles": "股四頭肌",
            "notes": ""
        }

    # ---------------------------------------------------------
    # 10. 上肢推：胸、肩、三頭
    # ---------------------------------------------------------
    if any(k in name for k in ["三頭肌", "伸臂"]):
        return {
            "standard_name": "三頭肌伸展訓練器",
            "main_category": "固定式阻力訓練器械",
            "sub_category": "上肢推－三頭肌",
            "resistance_type": "插銷配重／槓桿式",
            "primary_muscles": "肱三頭肌",
            "notes": ""
        }

    if any(k in name for k in ["側平舉", "側肩平舉", "側舉"]):
        return {
            "standard_name": "側平舉訓練機",
            "main_category": "固定式阻力訓練器械",
            "sub_category": "上肢推－肩部",
            "resistance_type": "插銷配重／槓桿式",
            "primary_muscles": "三角肌中束",
            "notes": ""
        }

    if any(k in name for k in ["推肩", "推膊", "肩上推舉", "肩膀推舉", "肩部推舉", "肩推舉"]):
        return {
            "standard_name": "肩部推舉訓練機",
            "main_category": "固定式阻力訓練器械",
            "sub_category": "上肢推－肩部",
            "resistance_type": "插銷配重／槓桿式",
            "primary_muscles": "三角肌／肱三頭肌",
            "notes": ""
        }

    if any(k in name for k in ["飛鳥", "蝴蝶", "夾胸", "擴胸"]):
        if any(k in name for k in ["後背", "背肌", "反式", "後三角"]):
            std = "胸式／反式飛鳥訓練機"
            sub = "上肢推及拉－飛鳥／反向飛鳥"
            muscles = "胸大肌／後三角肌／中背"
            note = "雙功能器材。"
        else:
            std = "蝴蝶機／飛鳥機"
            sub = "上肢推－胸部飛鳥"
            muscles = "胸大肌／前三角肌"
            note = ""

        return {
            "standard_name": std,
            "main_category": "固定式阻力訓練器械",
            "sub_category": sub,
            "resistance_type": "插銷配重／槓桿式",
            "primary_muscles": muscles,
            "notes": note
        }

    if any(k in name for k in ["推胸", "健胸", "胸推", "胸部推舉"]):
        return {
            "standard_name": "坐姿推胸機",
            "main_category": "固定式阻力訓練器械",
            "sub_category": "上肢推－胸部",
            "resistance_type": "插銷配重／槓桿式",
            "primary_muscles": "胸大肌／肱三頭肌／前三角肌",
            "notes": ""
        }

    # ---------------------------------------------------------
    # 11. 上肢拉：背、二頭、下拉、划船
    # ---------------------------------------------------------
    if any(k in name for k in ["二頭肌", "屈臂", "彎舉"]):
        return {
            "standard_name": "二頭肌彎舉訓練器",
            "main_category": "固定式阻力訓練器械",
            "sub_category": "上肢拉－二頭肌",
            "resistance_type": "插銷配重／槓桿式",
            "primary_muscles": "肱二頭肌／肱肌",
            "notes": ""
        }

    if any(k in name for k in ["滑輪下拉", "背闊肌下拉", "高拉", "上背肌拉力", "高拉力", "拉背"]):
        return {
            "standard_name": "高拉下拉訓練機",
            "main_category": "固定式阻力訓練器械",
            "sub_category": "上肢拉－垂直下拉",
            "resistance_type": "插銷配重／滑輪",
            "primary_muscles": "背闊肌／大圓肌／肱二頭肌",
            "notes": ""
        }

    if any(k in name for k in ["坐姿划船", "划船背肌", "中拉", "背肌拉力", "背拉"]):
        return {
            "standard_name": "坐姿划船訓練機",
            "main_category": "固定式阻力訓練器械",
            "sub_category": "上肢拉－水平划船",
            "resistance_type": "插銷配重／滑輪",
            "primary_muscles": "背闊肌／菱形肌／中斜方肌／肱二頭肌",
            "notes": ""
        }

    # ---------------------------------------------------------
    # 12. 史密夫機、訓練架
    # ---------------------------------------------------------
    if "史密夫" in name:
        return {
            "standard_name": "史密夫機",
            "main_category": "自由重量支架及訓練架",
            "sub_category": "導軌式槓鈴訓練",
            "resistance_type": "槓鈴／槓片",
            "primary_muscles": "全身",
            "notes": ""
        }

    if any(k in name for k in ["牆架", "牆梯", "攀牆架", "攀牆梯", "橫架"]):
        return {
            "standard_name": "牆架／攀牆梯",
            "main_category": "自由重量支架及訓練架",
            "sub_category": "伸展／自重訓練架",
            "resistance_type": "自體重",
            "primary_muscles": "全身",
            "notes": ""
        }

    # ---------------------------------------------------------
    # 13. 未能可靠判斷：列作待覆核
    # ---------------------------------------------------------
    return {
        "standard_name": name,
        "main_category": "待覆核",
        "sub_category": "未能由名稱可靠判斷",
        "resistance_type": "待確認",
        "primary_muscles": "待確認",
        "notes": "建議以器材相片、製造商型號或場館實物確認。"
    }


# 產生新欄位
records = df["original_name"].apply(make_record)
output_df = pd.concat([df, pd.DataFrame(records.tolist())], axis=1)

# 排序，令相同類別及標準名稱排在一起
output_df = output_df.sort_values(
    by=["main_category", "sub_category", "standard_name", "original_name"]
).reset_index(drop=True)

# 輸出完整 CSV；utf-8-sig 可讓 Excel 正確顯示中文
output_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

# 輸出需人工覆核項目
review_df = output_df[output_df["main_category"] == "待覆核"].copy()
review_df.to_csv(REVIEW_FILE, index=False, encoding="utf-8-sig")

print(f"已輸出：{OUTPUT_FILE}")
print(f"需覆核項目：{REVIEW_FILE}")
print(f"總項目數：{len(output_df)}")
print(f"待覆核數：{len(review_df)}")

已輸出：facility_names_standardized.csv
需覆核項目：facility_names_review_needed.csv
總項目數：581
待覆核數：89
